### NOTE: If you are reading this in our github repo and the project is done then you can simply take the converted and collated dataset that these scripts would make from our public google drive link here as it is far too large to put up on github (anyone with the link should be able to download): https://drive.google.com/file/d/1CNoh4WDAbwrwLNofJ_xqJxGaygTuhNMx/view?usp=share_link 

## With the above transformed and collated files you would no longer need to run these scripts

### MAIN HAM10000 DATASET: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
### HAM10k IMAGE WITH MASK SOURCE: https://www.kaggle.com/code/arianaliakbary/skin-cancer-lesions-segmentation-unet/input
### Transformation logic source: https://github.com/tamaraabuhawileh/Skin-Cancer-Object-Detection-YOLO/blob/main/Skin%20Cancer%20Object%20Detection%20Data%20Preperation/mask-to-bbox-yolo.ipynb

In [ ]:
import os

directory = 'data/labels'
os.makedirs(directory)

In [ ]:
import numpy as np
import cv2
import os
import pandas as pd

def mask_to_bounding_box(image_path, df, output_dir="data/labels"):
    try:
        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if image is None:
            print(f"Warning: Could not read image {image_path}. Skipping.")
            return

        # calc bounding boxes
        non_zero_indices = np.nonzero(image)
        if len(non_zero_indices[0]) == 0:
            print(f"Warning: No mask found in {image_path}. Skipping.")
            return
        
        min_y = np.min(non_zero_indices[0])
        max_y = np.max(non_zero_indices[0])
        min_x = np.min(non_zero_indices[1])
        max_x = np.max(non_zero_indices[1])

        # making dimensions & pixel coords to meet YOLO requiremnets
        img_height, img_width = image.shape[:2]
        box_width = max_x - min_x
        box_height = max_y - min_y
        center_x = min_x + box_width / 2
        center_y = min_y + box_height / 2

        #normalize
        norm_center_x = center_x / img_width
        norm_center_y = center_y / img_height
        norm_width = box_width / img_width
        norm_height = box_height / img_height
        
        # setting class ids
        img_name = os.path.basename(image_path).split('.')[0]
        mapping = {'MEL': 0, 'NV': 1, 'BCC': 2, 'AKIEC': 3, 'BKL': 4, 'DF': 5, 'VASC': 6}
        row = df[df['image'] == img_name]
        if row.empty:
            print(f"Warning: No metadata found for {img_name}. Skipping.")
            return
        clss = row.columns[(row == 1).iloc[0]][0]
        class_ID = mapping[clss]

        # save
        bounding_box_details = f"{class_ID} {norm_center_x} {norm_center_y} {norm_width} {norm_height}\n"
        
        os.makedirs(output_dir, exist_ok=True)
        output_file = os.path.join(output_dir, f"{img_name}.txt")
        
        with open(output_file, "w") as file:
            file.write(bounding_box_details)

    except Exception as e:
        print(f"An error occurred while processing {image_path}: {e}")


def process_all_masks(mask_folder, metadata_csv_path, output_label_dir):
    print("Loading metadata...")
    df = pd.read_csv(metadata_csv_path)
    
    mask_files = [f for f in os.listdir(mask_folder) if f.endswith('.png')]
    print(f"Found {len(mask_files)} masks to process.")
    
    for i, filename in enumerate(mask_files):
        if (i + 1) % 100 == 0:
            print(f"Processing file {i+1}/{len(mask_files)}...")
        img_path = os.path.join(mask_folder, filename)
        mask_to_bounding_box(img_path, df, output_label_dir)
        
    print("--- All masks processed! ---")


# The paths here are simply where i saved the masks and metadata -can change
MASK_DIRECTORY = "data/masks"
METADATA_FILE = "data/metadata.csv"
OUTPUT_LABEL_DIRECTORY = "data/labels"

process_all_masks(MASK_DIRECTORY, METADATA_FILE, OUTPUT_LABEL_DIRECTORY)

Loading metadata...
Found 10015 masks to process.
Processing file 100/10015...
Processing file 200/10015...
Processing file 300/10015...
Processing file 400/10015...
Processing file 500/10015...
Processing file 600/10015...
Processing file 700/10015...
Processing file 800/10015...
Processing file 900/10015...
Processing file 1000/10015...
Processing file 1100/10015...
Processing file 1200/10015...
Processing file 1300/10015...
Processing file 1400/10015...
Processing file 1500/10015...
Processing file 1600/10015...
Processing file 1700/10015...
Processing file 1800/10015...
Processing file 1900/10015...
Processing file 2000/10015...
Processing file 2100/10015...
Processing file 2200/10015...
Processing file 2300/10015...
Processing file 2400/10015...
Processing file 2500/10015...
Processing file 2600/10015...
Processing file 2700/10015...
Processing file 2800/10015...
Processing file 2900/10015...
Processing file 3000/10015...
Processing file 3100/10015...
Processing file 3200/10015...

In [ ]:
masks_directory = "data/masks"

for filename in os.listdir(masks_directory):
    file_path = os.path.join(masks_directory, filename)
    mask_to_bounding_box(file_path)

In [ ]:
masks_directory = "data/masks"
mask_count = 0
img_directory = "data/images"
img_count = 0
label_directory = "data/labels"
label_count = 0
mask_outlier_names = []
img_outlier_names = []
label_outlier_names = []

# loop & check if files are as expected
for file in os.scandir(masks_directory):
    if file.is_file() and (".png" in file.name):
        mask_count+=1
    else:
        mask_outlier_names.append(file.name)

for file in os.scandir(img_directory):
    if file.is_file() and (".jpg" in file.name):
        img_count+=1
    else:
        img_outlier_names.append(file.name)

for file in os.scandir(label_directory):
    if file.is_file() and (".txt" in file.name):
        label_count+=1
    else:
        label_outlier_names.append(file.name)

print("Total images: "+ str(img_count))
print("Outlier images: "+ str(img_outlier_names))
print("Total masks: "+ str(mask_count))
print("Outlier masks: "+ str(mask_outlier_names))
print("Total labels: "+ str(label_count))
print("Outlier labels: "+ str(label_outlier_names))

Total images: 10015
Outlier images: ['ATTRIBUTION.txt', 'LICENSE.txt']
Total masks: 10015
Outlier masks: []
Total labels: 10015
Outlier labels: []
